## 1. Importar bibliotecas

In [1]:
import pandas as pd


## 2. Ler a tabela

O arquivo Excel tem algumas linhas de título antes do cabeçalho de verdade.
Por isso usamos `header=2` (a 3ª linha do arquivo, contando a partir de 0) para pular o título e pegar os nomes das colunas corretos.


In [2]:
caminho_arquivo = "dados/CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"

df = pd.read_excel(caminho_arquivo, header=2)


df = df.dropna(axis=1, how="all")

df.head()


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
0,RO,11.0,15.0,Alta Floresta D'Oeste,24392.0,24392.0,21494.0
1,RO,11.0,23.0,Ariquemes,90353.0,90353.0,96833.0
2,RO,11.0,31.0,Cabixi,6313.0,6313.0,5351.0
3,RO,11.0,49.0,Cacoal,78574.0,78574.0,86887.0
4,RO,11.0,56.0,Cerejeiras,17029.0,17029.0,15890.0


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 7 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   UF                                                5572 non-null   str    
 1   COD. UF                                           5570 non-null   float64
 2   COD. MUNIC                                        5570 non-null   float64
 3   NOME DO MUNICÍPIO                                 5570 non-null   str    
 4   População Município 2010
(Sinopse)                5570 non-null   float64
 5   População 2010 (Alterações de Limites até 2022)1  5570 non-null   float64
 6   População Censo 2022                              5570 non-null   float64
dtypes: float64(5), str(2)
memory usage: 305.0 KB


## 3. Renomear colunas (deixar mais fácil de usar)

O arquivo traz duas colunas de 2010 (sinopse e "compatibilizada" com os limites de 2022).
Vamos usar a coluna **compatibilizada**, que é a correta para comparar com 2022 (mesmos limites municipais).


In [4]:
df = df.rename(columns={
    "UF": "UF",
    "COD. UF": "COD_UF",
    "COD. MUNIC": "COD_MUNIC",
    "NOME DO MUNICÍPIO": "MUNICIPIO",
    "População Município 2010\n(Sinopse)": "POP_2010_SINOPSE",
    "População 2010 (Alterações de Limites até 2022)1": "POP_2010",
    "População Censo 2022": "POP_2022",
})

df["POP_2010"] = pd.to_numeric(df["POP_2010"], errors="coerce")
df["POP_2022"] = pd.to_numeric(df["POP_2022"], errors="coerce")

df.head()


,UF,COD_UF,COD_MUNIC,MUNICIPIO,POP_2010_SINOPSE,POP_2010,POP_2022
0,RO,11.0,15.0,Alta Floresta D'Oeste,24392.0,24392.0,21494.0
1,RO,11.0,23.0,Ariquemes,90353.0,90353.0,96833.0
2,RO,11.0,31.0,Cabixi,6313.0,6313.0,5351.0
3,RO,11.0,49.0,Cacoal,78574.0,78574.0,86887.0
4,RO,11.0,56.0,Cerejeiras,17029.0,17029.0,15890.0


## 4. Tabela agregada por Estado (UF)

Somamos a população 2010 e 2022 de todos os municípios de cada estado.


In [5]:
pop_estado = (
    df.groupby("UF")[["POP_2010", "POP_2022"]]
    .sum()
    .reset_index()
)

pop_estado.head()


,UF,POP_2010,POP_2022
0,AC,733559.0,830018.0
1,AL,3120887.0,3127683.0
2,AM,3483985.0,3941613.0
3,AP,669526.0,733759.0
4,BA,14017071.0,14141626.0


## 5. Calcular o crescimento e ordenar (por estado)

Crescimento = População 2022 - População 2010.
Ordenamos do estado que mais cresceu para o que menos cresceu.


In [6]:
pop_estado["CRESCIMENTO"] = pop_estado["POP_2022"] - pop_estado["POP_2010"]
pop_estado["CRESCIMENTO_%"] = (pop_estado["CRESCIMENTO"] / pop_estado["POP_2010"] * 100).round(2)

pop_estado = pop_estado.sort_values("CRESCIMENTO", ascending=False).reset_index(drop=True)

pop_estado


,UF,POP_2010,POP_2022,CRESCIMENTO,CRESCIMENTO_%
0,SP,41262199.0,44411238.0,3149039.0,7.63
1,SC,6248436.0,7610361.0,1361925.0,21.80
2,GO,6001789.0,7056495.0,1054706.0,17.57
3,PR,10444526.0,11444380.0,999854.0,9.57
4,MG,19597330.0,20539989.0,942659.0,4.81
5,MT,3035122.0,3658649.0,623527.0,20.54
6,PA,7581051.0,8120131.0,539080.0,7.11
7,AM,3483985.0,3941613.0,457628.0,13.14
8,CE,8451644.0,8794957.0,343313.0,4.06
9,ES,3514952.0,3833712.0,318760.0,9.07


## 6. Salvar a tabela por estado em CSV

In [7]:
pop_estado.to_csv("crescimento_populacional_por_estado.csv", sep=";", index=False)
print("Arquivo salvo: crescimento_populacional_por_estado.csv")


Arquivo salvo: crescimento_populacional_por_estado.csv


## 7. Agora por Município

Como o arquivo já vem por município, não é necessário agrupar (cada linha já é um município).
Mesmo assim, usamos `groupby` para garantir que não existam municípios duplicados na base
(por exemplo, casos em que o mesmo código de município aparece mais de uma vez).


In [8]:
pop_municipio = (
    df.groupby(["UF", "COD_MUNIC", "MUNICIPIO"])[["POP_2010", "POP_2022"]]
    .sum()
    .reset_index()
)

pop_municipio.head()


,UF,COD_MUNIC,MUNICIPIO,POP_2010,POP_2022
0,AC,13.0,Acrelândia,12538.0,14021.0
1,AC,54.0,Assis Brasil,6072.0,8100.0
2,AC,104.0,Brasiléia,21398.0,26000.0
3,AC,138.0,Bujari,8471.0,12917.0
4,AC,179.0,Capixaba,8798.0,10392.0


## 8. Calcular o crescimento e ordenar (por município)


In [9]:
pop_municipio["CRESCIMENTO"] = pop_municipio["POP_2022"] - pop_municipio["POP_2010"]
pop_municipio["CRESCIMENTO_%"] = (pop_municipio["CRESCIMENTO"] / pop_municipio["POP_2010"] * 100).round(2)

pop_municipio = pop_municipio.sort_values("CRESCIMENTO", ascending=False).reset_index(drop=True)

pop_municipio.head(20)


,UF,COD_MUNIC,MUNICIPIO,POP_2010,POP_2022,CRESCIMENTO,CRESCIMENTO_%
0,AM,2603.0,Manaus,1802014.0,2063689.0,261675.0,14.52
1,DF,108.0,Brasília,2572159.0,2817381.0,245222.0,9.53
2,SP,50308.0,São Paulo,11253503.0,11451999.0,198496.0,1.76
3,SP,52205.0,Sorocaba,586816.0,723682.0,136866.0,23.32
4,GO,8707.0,Goiânia,1301912.0,1437366.0,135454.0,10.40
5,RR,100.0,Boa Vista,284313.0,413486.0,129173.0,45.43
6,SC,5407.0,Florianópolis,421240.0,537211.0,115971.0,27.53
7,PA,5536.0,Parauapebas,153908.0,267836.0,113928.0,74.02
8,MS,2704.0,Campo Grande,786774.0,898100.0,111326.0,14.15
9,PB,7507.0,João Pessoa,723515.0,833932.0,110417.0,15.26


## 9. Salvar a tabela por município em CSV

In [10]:
pop_municipio.to_csv("crescimento_populacional_por_municipio.csv", sep=";", index=False)
print("Arquivo salvo: crescimento_populacional_por_municipio.csv")


Arquivo salvo: crescimento_populacional_por_municipio.csv


## Conclusão

- `crescimento_populacional_por_estado.csv`: população 2010 x 2022, crescimento absoluto e percentual, por UF, ordenado do maior para o menor crescimento.
- `crescimento_populacional_por_municipio.csv`: mesma análise, mas por município.
